# 🔢 02-机器学习 · 推导笔记 2：逻辑回归（手写全流程）

> 逻辑回归是**面试手推之王**：sigmoid 为什么、交叉熵为什么、梯度为什么是 $X^{\top}(p-y)$——本篇全推导 + 手写。
> 核心实现全部手写（numpy 数值原语）；sklearn 仅用于对照验证（对照 cell 会显式标注）。

> 💡 一句话类比：预测「垃圾邮件概率」就像博彩公司定赔率——把赢/输比（odds）取对数后做线性打分，所以逻辑回归本质是「在 log-odds 空间里做线性回归」（§1 详解）。

**运行环境**：Python 3.8+，仅依赖 numpy（对照 cell 需要 sklearn）。

## 📖 本章导读

> **这一章讲什么**：从「对数几率（log-odds）」一步步推出 sigmoid；解释为什么损失必须选交叉熵；**手推梯度为什么是 $X^\top(p-y)$**；最后讲清它与 softmax、神经网络输出层的关系。

**学完本章，你应该能**：
1. 从 $\ln\frac{p}{1-p}=w^\top x$ 推出 $p=\sigma(w^\top x)=\frac{1}{1+e^{-w^\top x}}$；
2. 三步推导交叉熵梯度 $\nabla L = X^\top(p-y)$；
3. 解释为什么不用 MSE（非凸 + 梯度含 $\sigma'$ 消失）；
4. 讲清决策边界为什么是超平面、LR 与线性回归/神经网络的关系。

> 📌 全书第 2 章，面试「手推之王」。阅读路径：本章导读 → 原理 · 推导 · 代码 → 自测清单 → 本章小结。

## §1 为什么用 sigmoid：从线性模型到概率

分类任务需要输出 $[0,1]$ 的概率。对线性得分 $z = x^{\top}w$ 做挤压：

$$p(y{=}1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-z}}$$

**对数几率（logit）视角**：

$$\log\frac{p}{1-p} = z = x^{\top}w$$

把「几率比的对数」建模为线性——这就是「逻辑回归」名称的来源（对 logit 回归）。

> 💡 **「log-odds 下注」直觉**：博彩公司给你的球队开 3:1——意思赢面是输面的 3 倍（odds=3，即 $p=0.75$）。odds 是「赢/输」的比值，log-odds 就是它的对数。串关押注时多场要**乘**赔率，取对数后**乘法变加法**。逻辑回归正是把 log-odds 当作特征的线性组合：特征每加 1，log-odds 就加 $w_j$，等价于「赢面被乘以 $e^{w_j}$ 倍」。所以逻辑回归可以理解为：**在对数赔率空间里做线性回归**。

### sigmoid 的三个关键性质

$$\sigma(-z) = 1 - \sigma(z) \quad (\text{对称性})$$

$$\sigma'(z) = \sigma(z)\big(1 - \sigma(z)\big) \quad (\text{导数自洽，梯度推导核心})$$

**数值稳定性**：$z$ 很负时 $e^{-z}$ 溢出，用分段公式：

$$\sigma(z) = \begin{cases} \dfrac{1}{1+e^{-z}} & z \ge 0 \\\\ \dfrac{e^{z}}{1+e^{z}} & z < 0 \end{cases}$$

## §2 交叉熵损失的完整推导（面试必背）

**第一步**：写出似然。样本独立，$y_i \in \{0,1\}$：

$$L(w) = \prod_{i=1}^n p_i^{y_i} (1-p_i)^{1-y_i}, \quad p_i = \sigma(x_i^{\top}w)$$

**第二步**：取负对数（最大化似然 = 最小化负对数似然）：

$$-\log L(w) = -\sum_{i=1}^n \Big[ y_i \log p_i + (1-y_i)\log(1-p_i) \Big]$$

这就是**二分类交叉熵**。

**第三步**：求梯度。利用 $\sigma' = \sigma(1-\sigma)$：

$$\frac{\partial \ell_i}{\partial z_i} = -\left( \frac{y_i}{p_i} p_i(1-p_i) - \frac{1-y_i}{1-p_i} p_i(1-p_i) \right) = p_i - y_i$$

**第四步**：链式法则到 $w$：

$$\nabla_w L = \sum_{i=1}^n (p_i - y_i) x_i = X^{\top}(p - y)$$

> 💡 结论超漂亮：**梯度 = 预测误差 $\times$ 特征**。与线性回归 $X^{\top}(Xw-y)$ 形式完全一致——都是广义线性模型的统一梯度。

> ⚠️ **口径统一（求和版 vs 批平均版）**：上式是「求和版」。若损失取均值（`np.mean`），对应**批平均版**梯度 $\frac{1}{n} X^{\top}(p-y)$。两者只差一个常数倍：不除 $n$ 也行，等价于把学习率放大 $n$ 倍，收敛到**同一个最优点**。下方代码实现的是批平均版。

## §3 手写实现：二分类逻辑回归

**更新公式**（求和版）：

$$w_{t+1} = w_t - \eta \, X^{\top}(\sigma(Xw_t) - y)$$

**更新公式**（批平均版，对应下方代码）：

$$w_{t+1} = w_t - \frac{\eta}{n} \, X^{\top}(\sigma(Xw_t) - y)$$

> ⚠️ 代码里 `grad = Xb.T @ (p - y) + 2*lam*w` 后 `w -= lr * grad / n`，是**批平均版**（除以 n）。除以 n 与否只改变学习率的有效尺度，最优解不变——与 §2 公式口径统一。

**要点**：

- sigmoid 数值稳定版
- 加 L2 正则：梯度加 $2\lambda w$
- 收敛判据：loss 变化小于阈值或达到 epoch 上限

In [ ]:
import numpy as np

def sigmoid(z):
    # 数值稳定版：z>=0 用 1/(1+exp(-z))，z<0 用 exp(z)/(1+exp(z))
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    out[~pos] = np.exp(z[~pos]) / (1.0 + np.exp(z[~pos]))
    return out

def make_class_data(n=120, seed=0):
    rng = np.random.default_rng(seed)
    # 两个二维高斯簇，中心 (1,1) 与 (-1,-1)
    X = np.vstack([rng.normal([1, 1], 0.8, (n, 2)),
                   rng.normal([-1, -1], 0.8, (n, 2))])
    y = np.concatenate([np.ones(n), np.zeros(n)])
    return X, y

class LogisticRegression:
    def __init__(self, lr=0.5, epochs=2000, lam=0.0):
        self.lr = lr
        self.epochs = epochs
        self.lam = lam
        self.w = None
        self.loss_hist = []

    def fit(self, X, y):
        Xb = np.hstack([np.ones((len(X), 1)), X])  # 偏置列
        n, d = Xb.shape
        self.w = np.zeros(d)
        for _ in range(self.epochs):
            p = sigmoid(Xb @ self.w)
            grad = Xb.T @ (p - y) + 2 * self.lam * self.w
            self.w -= self.lr * grad / n
            eps = 1e-12
            loss = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
            self.loss_hist.append(loss)
        return self

    def predict_proba(self, X):
        Xb = np.hstack([np.ones((len(X), 1)), X])
        return sigmoid(Xb @ self.w)

    def predict(self, X, thr=0.5):
        return (self.predict_proba(X) >= thr).astype(int)

X, y = make_class_data()
model = LogisticRegression(lr=0.5, epochs=1000).fit(X, y)
pred = model.predict(X)
acc = np.mean(pred == y)
print('手写 LR 训练准确率 =', round(acc, 4))          # 期望约 0.98+
print('损失下降示例: %.4f -> %.4f' % (model.loss_hist[0], model.loss_hist[-1]))
print('权重(含偏置) w =', np.round(model.w, 3))      # 决策边界近似线性可分
print('sigmoid 数值稳定性检查: sigmoid(-800) =', sigmoid(-800), '（不溢出）')

In [ ]:
# ---- 可视化：sigmoid 曲线及其导数（运行本 cell 生成配图）----
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

z = np.linspace(-8, 8, 300)
s = sigmoid(z)
plt.figure(figsize=(7, 4.2))
plt.plot(z, s, "b-", lw=2, label="sigmoid σ(z)")
plt.plot(z, s * (1 - s), "r--", lw=2, label="导数 σ′(z) = σ(1−σ)")
plt.axhline(0.5, color="gray", ls=":", lw=1)
plt.axvline(0, color="gray", ls=":", lw=1)
plt.xlabel("z = wᵀx + b")
plt.ylabel("σ(z)")
plt.title("sigmoid 函数及其导数：z 越大越接近 1，导数在 z=0 处最大")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## §3.5 sklearn 对照：手写梯度下降 LR vs LogisticRegression

在相同 2D 数据上对比手写 LR 与 `sklearn.linear_model.LogisticRegression`：

- sklearn 用 `solver='lbfgs'`，并把正则强度调到极小（`C=1e10` ≈ 不正则），与手写 `lam=0` 对齐
- 对比指标：测试集准确率、预测一致率、决策边界法向量方向（余弦相似度）
- 权重**尺度**可能有微小差异（优化器收敛容差不同），但**方向**一致、决策边界等价——分类看的是边界，不是权重绝对值

In [ ]:
# ---- sklearn 对照：手写 GD LR vs sklearn LogisticRegression ----
try:
    from sklearn.linear_model import LogisticRegression as SkLR
    HAS_SK = True
except ImportError:
    HAS_SK = False

if HAS_SK:
    # 训练/测试分开（同分布、不同随机噪声），公平对比泛化能力
    X_tr, y_tr = make_class_data(n=150, seed=3)
    X_te, y_te = make_class_data(n=80, seed=7)

    # 手写：批量梯度下降（批平均版梯度），无正则
    hand = LogisticRegression(lr=0.5, epochs=2000, lam=0.0).fit(X_tr, y_tr)
    # sklearn：极高 C ≈ 无正则，与手写 lam=0 对齐
    sklr = SkLR(C=1e10, solver='lbfgs', max_iter=5000).fit(X_tr, y_tr)

    acc_hand = np.mean(hand.predict(X_te) == y_te)
    acc_sk = sklr.score(X_te, y_te)
    agree = np.mean(hand.predict(X_te) == sklr.predict(X_te))

    # 决策边界法向量方向一致性：边界由 w 的方向（而非大小）决定
    v_hand = hand.w[1:] / np.linalg.norm(hand.w[1:])
    v_sk = sklr.coef_[0] / np.linalg.norm(sklr.coef_[0])
    cos = float(v_hand @ v_sk)

    print('手写 LR 测试准确率 =', round(acc_hand, 4))
    print('sklearn LR 测试准确率 =', round(acc_sk, 4))
    print('手写 vs sklearn 预测一致率 =', round(agree, 4))
    print('决策边界法向量余弦相似度 =', round(cos, 4))
    print('手写权重   w =', np.round(hand.w, 3))
    print('sklearn 权重 =', np.round(np.concatenate([[sklr.intercept_[0]], sklr.coef_[0]]), 3))
    assert agree > 0.95 and cos > 0.99, '手写 LR 与 sklearn 决策不一致'
    print('✓ 手写 LR 与 sklearn 决策边界一致')
else:
    print('未安装 sklearn，跳过对照（不影响手写部分）')

## §4 softmax 多分类手写

**多分类**：$K$ 个类别，每类一个权重向量 $w_k$，得分 $z_k = x^{\top}w_k$：

$$p_k = \operatorname{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

**数值稳定版**（减最大值）：

$$p_k = \frac{e^{z_k - \max z}}{\sum_j e^{z_j - \max z}}$$

**交叉熵 + 梯度**（one-hot $y$）：

$$L = -\sum_i \sum_k y_{ik} \log p_{ik}$$

$$\frac{\partial L}{\partial z_k} = p_k - y_k \quad \Rightarrow \quad \nabla_{W} L = X^{\top}(P - Y)$$

**推导核心**：$\dfrac{\partial p_k}{\partial z_j} = p_k(\delta_{kj} - p_j)$（softmax 的雅可比），链式法则后只剩 $p_k - y_k$。

In [ ]:
def softmax(Z):
    # Z: (n, K) 原始得分；行内减最大值防溢出；先转 ndarray 兼容 list 输入
    Z = np.asarray(Z, dtype=float)
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def softmax_regression(X, y, K, lr=0.5, epochs=1000, seed=0):
    rng = np.random.default_rng(seed)
    Xb = np.hstack([np.ones((len(X), 1)), X])
    n, d = Xb.shape
    W = rng.normal(0, 0.01, (d, K))
    Y = np.eye(K)[y]                       # one-hot
    for _ in range(epochs):
        P = softmax(Xb @ W)
        grad = Xb.T @ (P - Y) / n           # 手写梯度 X^T(P - Y)
        W -= lr * grad
    return W

# 三类数据
rng = np.random.default_rng(1)
means = [[2, 0], [0, 2], [-2, -2]]
Xm = np.vstack([rng.normal(m, 0.6, (60, 2)) for m in means])
ym = np.concatenate([np.zeros(60), np.ones(60), np.full(60, 2)]).astype(int)

W = softmax_regression(Xm, ym, K=3)
Xb = np.hstack([np.ones((len(Xm), 1)), Xm])
pred = softmax(Xb @ W).argmax(axis=1)
acc = np.mean(pred == ym)
print('手写 softmax 回归训练准确率 =', round(acc, 4))   # 期望接近 1.0
print('softmax 行和 = 1 检查:', np.allclose(softmax(Xb @ W).sum(axis=1), 1))
print('softmax 数值稳定性: softmax([1000, 1001]) ≈', np.round(softmax([[1000, 1001]]), 6))

## §5 决策边界 / 类别不平衡 / 正则

### 决策边界

二分类边界 $p = 0.5 \iff z = 0$，即 $x^{\top}w = 0$——高维空间一个**超平面**。

### 类别不平衡

- 重采样（上采样少数类 / 下采样多数类）
- 代价敏感：损失里少数类样本权重 $w_i > 1$
- 阈值调整：预测概率 > 0.5 不一定最优，用验证集选阈值

### 正则化

$$L(w) = \text{cross-entropy} + \lambda\|w\|_2^2 \quad (\text{或 } \lambda\|w\|_1)$$

高维稀疏特征（CTR 场景）常用 L1 产生稀疏权重。

## §6 常见 bug 与边界（面试踩坑区）

- **sigmoid 溢出**：直接 `1/(1+np.exp(-z))` 在 z=-800 时 exp(800) 溢出 → 用分段公式
- **softmax 溢出**：先减行最大值再 exp
- **log(0)**：p 恰好为 0/1 时 loss 变 inf/NaN → 加 `eps=1e-12`
- **one-hot 索引错误**：`np.eye(K)[y]` 中 y 必须是整数 0..K-1
- **梯度口径混乱**：求和版 $X^{\top}(p-y)$ 与批平均版 $X^{\top}(p-y)/n$ 混用会让有效学习率差 n 倍（§2/§3 已统一说明）
- **收敛检查**：只看 loss 最后值不看趋势；应打印 loss_hist 前中后
- **类别不平衡没处理**：直接训练导致预测全偏向多数类
- **lr 过大 loss 变 NaN**：梯度带 $X^{\top}(p-y)$，特征未标准化时数值不稳定——标准化后再训

## §7 面试追问与扩展（加分项）

### 7.1 LR vs SVM（高频对比题）

| 维度 | 逻辑回归 | SVM |
|------|---------|-----|
| 输出 | 概率（可校准） | 距离/符号，概率需 Platt |
| 损失 | 对数损失（光滑） | 合页损失（分段） |
| 边界 | 所有样本参与 | **仅支持向量**决定 |
| 核技巧 | 需显式特征 | 核函数隐式升维 |
| 小样本 | 一般 | 更鲁棒（间隔最大化） |

### 7.2 为什么用交叉熵而不是 MSE

MSE + sigmoid 时梯度含 $\sigma'(z)$，接近饱和区梯度消失，收敛慢；交叉熵让梯度 = $p-y$，与概率差值成正比，永不消失。

### 7.3 广义线性模型统一视角

线性回归（恒等链接）、逻辑回归（logit 链接）、softmax（多分类）——**同一个框架：线性得分 + 链接函数 + 对应损失**。

### 7.4 与 ML 后期模型

LR 是**神经网络最后一层**的原型（输出层 sigmoid/softmax + 交叉熵）；GBDT+LR、DeepFM 等工业模型都以 LR 思想为基础。

## §8 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 写出 sigmoid 公式与三个性质（对称、导数、数值稳定）
- [ ] 2. 从似然推导交叉熵损失
- [ ] 3. 推导梯度 $\nabla_w L = X^{\top}(p - y)$
- [ ] 4. 手写二分类 LR（numpy，不调 sklearn）
- [ ] 5. 手写稳定 sigmoid / softmax
- [ ] 6. 说明决策边界为什么是超平面
- [ ] 7. 手写准确率 / 精确率 / 召回率 / F1
- [ ] 8. 说明类别不平衡的三种处理
- [ ] 9. 解释 logit 与概率的关系
- [ ] 10. 手写预测函数 predict / predict_proba

### L2 进阶（10 问）

- [ ] 11. 推导 softmax 的雅可比 $\partial p_k/\partial z_j$
- [ ] 12. 推导 softmax 交叉熵梯度 $p_k - y_k$
- [ ] 13. 手写 softmax 多分类回归
- [ ] 14. LR 加 L1/L2 正则后梯度变化
- [ ] 15. 手写 SGD 版 LR（逐样本）
- [ ] 16. 手写 mini-batch 版
- [ ] 17. 多分类 OVR 手写（K 个二分类）
- [ ] 18. 手写 AUC 计算（排序法）
- [ ] 19. 阈值选择（P-R 曲线手算）
- [ ] 20. 手写带校验集早停的 LR

### L3 挑战（5 问）

- [ ] 21. LR 的牛顿法 / IRLS 推导
- [ ] 22. FTRL 在线学习（广告 CTR 场景）手写
- [ ] 23. 多分类与二分类关系的形式化证明（softmax 等价 K-1 个二分类）
- [ ] 24. 逻辑回归的校准（Platt / 温度缩放）
- [ ] 25. 用 LR 做特征重要性归因（权重 vs odds 解释）

### 自测要点

- 10 分钟内盲写：二分类 LR 完整类（numpy）
- 白板推导：交叉熵 → 梯度 $X^{\top}(p-y)$ 全链路
- 对比 LR vs SVM 能讲 3 分钟

## 附录：参考与下节预告

- 《统计学习方法》第 6 章（逻辑回归与最大熵）
- PRML 第 4 章（广义线性模型）

### 下节预告：推导笔记 3 — 朴素贝叶斯 + K-Means（生成式模型与聚类）

> 💡 本笔记验收：`02-机器学习/README.md` 验收清单「LR / SVM / PCA 必含」中 LR 篇打勾。

## 🏁 本章小结

- **原理一句话**：逻辑回归 = **在 log-odds 空间做线性回归**，输出概率。
- **必会公式**：$p=\sigma(w^\top x)=\frac{1}{1+e^{-w^\top x}}$；$\sigma'=\sigma(1-\sigma)$；$\nabla L=X^\top(p-y)$。
- **面试怎么考**：三步梯度推导；为什么不用 MSE（非凸 + 梯度含 $\sigma'$ 消失）；决策边界为什么是超平面；LR = 最大熵特例 = 神经网络最后一层原型。
- **易错点**：sigmoid 数值溢出（用分段稳定版）；梯度记成 $p-y$ 却漏了 $X^\top$。

> 自测清单没把握的条目，回到对应小节重读后再打勾。